# SMS backup cleanup (runs inside *your own* Google account)

Each person runs this notebook while signed in to the Google account that owns the backups.
Nothing is downloaded to your computer and nothing leaves your Google account: Colab runs in Google's cloud and reads your Drive directly.

**Steps:** 1) mount Drive → 2) set the folder → 3) `analyze` (read-only, re-run until finished) → 4) read the report → 5) `apply` → 6) delete the `superseded` folder and **empty the Drive trash**.

Tip: *Runtime → Change runtime type* is not needed; the free CPU runtime is fine. If Colab disconnects, just run the cells again — progress is saved on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the tool. If the repo is private, use a fine-grained read-only token:
#   !git clone https://<TOKEN>@github.com/LegitUnicorn/SMS-Cleanup.git
!rm -rf /content/SMS-Cleanup && git clone -q https://github.com/LegitUnicorn/SMS-Cleanup.git /content/SMS-Cleanup
%cd /content/SMS-Cleanup

In [ ]:
# ---- EDIT THIS: the Drive folder that holds sms-YYYYMMDDHHMMSS.xml / calls-*.xml ----
BACKUP_DIR = '/content/drive/MyDrive/SMSBackupRestore'
WORK_DIR   = BACKUP_DIR + '-cleanup'   # state, report, recovered + superseded files

import os, glob
files = sorted(glob.glob(os.path.join(BACKUP_DIR, '*.xml')))
total = sum(os.path.getsize(f) for f in files)
print(f'{len(files)} xml files, {total/1e9:.1f} GB')
for f in files[:5] + (['...'] if len(files) > 10 else []) + files[-5:]:
    print('  ', os.path.basename(f))

In [ ]:
# Optional sanity check on the newest backup (counts + date range)
newest = max(glob.glob(os.path.join(BACKUP_DIR, 'sms-*.xml')))
!python -m smscleanup inspect "$newest"

## Analyze (read-only for your backup folder)
Reads every backup once, newest → oldest. Safe to stop and re-run at any time; it continues where it left off.
`--max-hours 10` keeps it inside a Colab session; just run it again afterwards.

In [ ]:
!python -m smscleanup analyze "$BACKUP_DIR" --work-dir "$WORK_DIR" --max-hours 10 --tmp-dir /content/tmp

In [ ]:
from IPython.display import Markdown
Markdown(open(os.path.join(WORK_DIR, 'report.md')).read())

## Apply
Before touching anything this re-verifies, from disk, that every message in every file it is about to remove is present in a kept file or in `recovered/`.
Superseded files are **moved** to `WORK_DIR/superseded` (not deleted) so you can look before you delete.

In [ ]:
!python -m smscleanup apply "$BACKUP_DIR" --work-dir "$WORK_DIR" --yes

## Free the space
1. In drive.google.com open the `…-cleanup/superseded` folder, select all, delete.
2. Open **Trash → Empty trash**. Storage is only released after this (it can take a few hours for the quota meter to update).
3. **Keep** `…-cleanup/recovered/*.xml` — those hold messages that were in old backups only (e.g. deleted from the phone later). They are valid backup files you can restore with the app.